### 1) 라이브러리 설치

In [ ]:
# uv add pypdf=">=4.2.0,<5.0.0"
# uv add langchain-upstage

#### PyPDFLoader 간단한 예제

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[:5])

up_gq


#### Upstage 를 사용한 RAG
* ChatUpstage
* UpstageEmbeddings
* UpstageDocumentParseLoader

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings

print("==> 1. 문서 로딩 → PDF 읽기...")
file_path = "../data/tutorial-korean.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(f"  총 {len(documents)}페이지 로드 완료")


==> 1. 문서 로딩 → PDF 읽기...
  총 39페이지 로드 완료


In [4]:

print("==> 2. 문서 분할 → 작은 청크로 나누기")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # 청크 크기 (한국어 최적화)
    chunk_overlap=200,      # 중복 부분 (맥락 보존)
    separators=["\n\n", "\n", ".", " ", ""] # 자연스러운 분할을 위한 구분자
)

chunks = text_splitter.split_documents(documents)
print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f}자")
print(type(chunks[0]))

==> 2. 문서 분할 → 작은 청크로 나누기
  66개 청크 생성 완료
  평균 청크 길이: 688자
<class 'langchain_core.documents.base.Document'>


In [5]:

print("==> 3. 벡터화 → 임베딩으로 변환")
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

print("==> 4. 저장 → FAISS 벡터스토어에 저장")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f" FAISS 벡터스토어 생성 완료 ({len(chunks)}개 벡터)")
# 로컬 파일로 저장
vectorstore.save_local("faiss_db")

print("===> 5. 검색 → 질문과 유사한 문서 찾기")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}  # 상위 6개 관련 문서 검색
)
print(" Retriever 설정 완료")

==> 3. 벡터화 → 임베딩으로 변환
==> 4. 저장 → FAISS 벡터스토어에 저장
 FAISS 벡터스토어 생성 완료 (66개 벡터)
===> 5. 검색 → 질문과 유사한 문서 찾기
 Retriever 설정 완료


In [6]:

print("===> 6. 생성 → LLM으로 답변 생성")
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

===> 6. 생성 → LLM으로 답변 생성
solar-pro


In [9]:


def format_docs(docs):
    """문서들을 문자열로 포맷"""
    return "\n\n".join(doc.page_content for doc in docs)

prompt_template = """
당신은 BlueJ 프로그래밍 환경 전문가입니다. 
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별 설명이 필요하면 순서대로 작성하세요  
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

답변:"""

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(prompt_template)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='\n당신은 BlueJ 프로그래밍 환경 전문가입니다. \n아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.\n\n문서 내용:\n{context}\n\n질문: {question}\n\n답변 규칙:\n1. 문서 내용만을 근거로 답변하세요\n2. 단계별 설명이 필요하면 순서대로 작성하세요  \n3. 구체적인 메뉴명, 버튼명을 포함하세요\n4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요\n\n답변:'


In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LCEL RAG 체인 (RetrievalQA 대체!)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 소스 문서 포함 버전
def rag_with_sources(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    
    response = llm.invoke(prompt.format(context=context, question=question))
    return {
        "answer": response.content,
        "source_documents": docs  # RetrievalQA와 동일!
    }

print("  LCEL RAG 파이프라인 구축 완료!")

  LCEL RAG 파이프라인 구축 완료!


In [11]:

# ===================================
# 테스트
# ===================================
test_questions = [
    "BlueJ에서 객체를 생성하는 방법은 무엇인가요?",
    "컴파일 오류가 발생했을 때 어떻게 확인할 수 있나요?", 
    "디버깅을 위해 중단점을 설정하는 방법을 알려주세요",
    "코드패드는 무엇이고 어떻게 사용하나요?",
    "애플릿을 만들고 실행하는 방법을 설명해주세요"
]

print("\n" + "=" * 60)
print(" LCEL RAG 시스템 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/5】")
    print(f"Q: {question}")
    
    # LCEL 체인 실행
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    
    # 소스 문서 확인 (RetrievalQA와 동일)
    docs = retriever.invoke(question)
    print(f"  참조: {len(docs)}개 문서")
    for j, doc in enumerate(docs[:2], 1):
        page = doc.metadata.get('page', '?')
        print(f"    {j}. P{page}: {doc.page_content[:80]}...")
    
    print("-" * 40)


 LCEL RAG 시스템 테스트

【테스트 1/5】
Q: BlueJ에서 객체를 생성하는 방법은 무엇인가요?
A: BlueJ에서 객체를 생성하는 방법은 다음과 같습니다. 문서 내용을 근거로 단계별로 설명드립니다:

1. **프로젝트 열기**  
   - BlueJ 실행 후 `Project - Open...` 메뉴를 선택하여 프로젝트를 엽니다.  
   - 예제 프로젝트인 `people` 프로젝트를 사용할 경우, `examples` 디렉토리에서 선택합니다(문서 83.2절 참조).

2. **클래스 아이콘에서 객체 생성**  
   - 메인 윈도우 중앙의 클래스 아이콘(예: `Database`, `Person`)에서 **마우스 오른쪽 버튼 클릭** (Mac: `Ctrl+Click`)합니다.  
   - 팝업 메뉴에서 **생성자 함수**를 선택합니다(문서 3.3절 참조).  
   - 대화상자가 나타나면 객체 이름을 입력(기본값 사용 가능)한 후 **OK 버튼**을 클릭합니다.  
   - 생성된 객체는 **오브젝트 벤치(Object Bench)**에 표시됩니다.

3. **추상 클래스 확인**  
   - 추상 클래스(예: `Person`)는 `<<abstract>>` 표기가 있으며, 객체 생성이 불가능합니다(문서 3.3절 마지막 부분).

4. **라이브러리 클래스의 객체 생성**  
   - JDK 라이브러리 클래스(예: `ArrayList`, `String`)의 객체를 생성하려면:  
     - `Tools - Use Libraries Class` 메뉴를 선택합니다.  
     - 패키지 이름을 포함한 **완전한 클래스명**(예: `java.lang.String`)을 입력 후 **엔터**를 누릅니다.  
     - 생성자 또는 정적 메소드 목록에서 선택하여 호출합니다(문서 3610.6절 참조).

> 요약: 일반 클래스는 클래스 아이콘의 팝업 메뉴에서 생성자를 선택하고, 라이브러리 클래스는 `Tools - Use Libraries Class` 메뉴를 사